In [1]:
# Cell 1 - NCBI E-utilities healthcheck

import time
import xml.etree.ElementTree as ET

import requests

EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
TIMEOUT_SECONDS = 20
REQUEST_DELAY_SECONDS = 0.35

session = requests.Session()
session.headers.update({"User-Agent": "pubmed-screening-assistant/0.1"})


def eutils_get(endpoint, params):
    response = session.get(f"{EUTILS}/{endpoint}.fcgi", params=params, timeout=TIMEOUT_SECONDS)
    response.raise_for_status()
    time.sleep(REQUEST_DELAY_SECONDS)
    return response


query = (
    'immunotherapy[Title/Abstract] AND cancer[Title/Abstract]'
    ' AND hasabstract[text] AND english[Language]'
    ' AND "journal article"[Publication Type]'
    ' AND ("2020/01/01"[Date - Publication] : "2026/12/31"[Date - Publication])'
)

search = eutils_get(
    "esearch",
    {
        "db": "pubmed",
        "term": query,
        "retmax": 2,
        "retmode": "json",
        "sort": "relevance",
    },
)
search_payload = search.json()["esearchresult"]
print(f"esearch_status={search.status_code} | total_hits={search_payload['count']} | ids_returned={len(search_payload['idlist'])}")
print(f"query_translation={search_payload.get('querytranslation', 'missing')}")

fetch = eutils_get(
    "efetch",
    {"db": "pubmed", "id": ",".join(search_payload["idlist"]), "retmode": "xml"},
)
root = ET.fromstring(fetch.content)
records = root.findall(".//PubmedArticle")
print(f"efetch_status={fetch.status_code} | records_parsed={len(records)}")

for record in records:
    pmid = record.findtext(".//MedlineCitation/PMID") or "missing"
    title_node = record.find(".//ArticleTitle")
    title = "".join(title_node.itertext()).strip() if title_node is not None else ""
    abstract = " ".join(
        "".join(node.itertext()).strip()
        for node in record.findall(".//Abstract/AbstractText")
    ).strip()
    print(f"pmid={pmid} | title_chars={len(title)} | abstract_words={len(abstract.split())}")

esearch_status=200 | total_hits=80499 | ids_returned=2
query_translation="immunotherapy"[Title/Abstract] AND "cancer"[Title/Abstract] AND "hasabstract"[Text Word] AND "english"[Language] AND "journal article"[Publication Type] AND 2020/01/01:2026/12/31[Date - Publication]


efetch_status=200 | records_parsed=2
pmid=34207103 | title_chars=55 | abstract_words=95
pmid=40847226 | title_chars=33 | abstract_words=172


In [2]:
# Cell 2 - ESearch per domain and persist PMID lists

import json
from pathlib import Path

OUTPUT_DIR = Path("data/raw/esearch")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RETMAX_PER_DOMAIN = 350

DOMAIN_TOPICS = {
    "immunotherapy_cancer": 'immunotherapy[Title/Abstract] AND cancer[Title/Abstract]',
    "cardiovascular_disease": '"cardiovascular disease"[Title/Abstract]',
}

SHARED_FILTERS = (
    " AND hasabstract[text] AND english[Language]"
    ' AND "journal article"[Publication Type]'
    ' AND ("2020/01/01"[Date - Publication] : "2026/12/31"[Date - Publication])'
)

retrieved_ids = {}
for domain, topic in DOMAIN_TOPICS.items():
    response = eutils_get(
        "esearch",
        {
            "db": "pubmed",
            "term": topic + SHARED_FILTERS,
            "retmax": RETMAX_PER_DOMAIN,
            "retmode": "json",
            "sort": "relevance",
        },
    )
    payload = response.json()["esearchresult"]
    raw_path = OUTPUT_DIR / f"esearch_{domain}.json"
    raw_path.write_text(json.dumps(response.json(), indent=2), encoding="utf-8")

    id_list = payload["idlist"]
    retrieved_ids[domain] = set(id_list)
    ids_path = OUTPUT_DIR / f"pmids_{domain}.txt"
    ids_path.write_text("\n".join(id_list) + "\n", encoding="utf-8")

    print(
        f"domain={domain} | total_hits={payload['count']} | ids_returned={len(id_list)}"
        f" | unique_ids={len(retrieved_ids[domain])} | raw_bytes={raw_path.stat().st_size}"
    )

domains = list(DOMAIN_TOPICS)
overlap = retrieved_ids[domains[0]] & retrieved_ids[domains[1]]
print(f"cross_domain_overlap={len(overlap)}")

domain=immunotherapy_cancer | total_hits=80499 | ids_returned=350 | unique_ids=350 | raw_bytes=7103


domain=cardiovascular_disease | total_hits=90378 | ids_returned=350 | unique_ids=350 | raw_bytes=7081
cross_domain_overlap=0


In [3]:
# Cell 3 - EFetch XML and parse records per domain

import csv
import xml.etree.ElementTree as ET
from pathlib import Path

PMID_DIR = Path("data/raw/esearch")
FETCH_DIR = Path("data/raw/efetch")
FETCH_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 200
FIELDS = ["pmid", "domain", "title", "abstract", "journal", "pub_year", "pub_types"]


def parse_article(article, domain):
    citation = article.find("MedlineCitation")
    pmid = citation.findtext("PMID") if citation is not None else None
    title_node = article.find(".//ArticleTitle")
    title = " ".join("".join(title_node.itertext()).split()) if title_node is not None else ""
    abstract_parts = []
    for node in article.findall(".//Abstract/AbstractText"):
        label = node.get("Label")
        text = " ".join("".join(node.itertext()).split())
        if text:
            abstract_parts.append(f"{label}: {text}" if label else text)
    journal = article.findtext(".//Journal/Title") or ""
    year = (
        article.findtext(".//JournalIssue/PubDate/Year")
        or article.findtext(".//JournalIssue/PubDate/MedlineDate")
        or ""
    )
    pub_types = [
        node.text for node in article.findall(".//PublicationTypeList/PublicationType") if node.text
    ]
    return {
        "pmid": pmid or "",
        "domain": domain,
        "title": title,
        "abstract": " ".join(abstract_parts),
        "journal": journal,
        "pub_year": year,
        "pub_types": ";".join(pub_types),
    }


records = []
for pmid_path in sorted(PMID_DIR.glob("pmids_*.txt")):
    domain = pmid_path.stem.removeprefix("pmids_")
    pmids = [line.strip() for line in pmid_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    fetched = 0
    for start in range(0, len(pmids), BATCH_SIZE):
        batch = pmids[start : start + BATCH_SIZE]
        response = eutils_get("efetch", {"db": "pubmed", "id": ",".join(batch), "retmode": "xml"})
        xml_path = FETCH_DIR / f"efetch_{domain}_{start:03d}.xml"
        xml_path.write_bytes(response.content)
        articles = ET.fromstring(response.content).findall(".//PubmedArticle")
        records.extend(parse_article(article, domain) for article in articles)
        fetched += len(articles)
        print(
            f"domain={domain} | batch_start={start} | requested={len(batch)}"
            f" | parsed={len(articles)} | xml_bytes={xml_path.stat().st_size}"
        )
    print(f"domain={domain} | total_requested={len(pmids)} | total_parsed={fetched}")

out_path = FETCH_DIR / "records_raw.csv"
with out_path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=FIELDS)
    writer.writeheader()
    writer.writerows(records)

word_counts = [len(record["abstract"].split()) for record in records]
print(
    f"records_total={len(records)} | unique_pmids={len({r['pmid'] for r in records})}"
    f" | empty_abstract={sum(1 for w in word_counts if w == 0)}"
)
print(
    f"abstract_words_min={min(word_counts)} | max={max(word_counts)}"
    f" | under_50={sum(1 for w in word_counts if w < 50)}"
)
print(f"csv={out_path} | csv_bytes={out_path.stat().st_size}")


domain=cardiovascular_disease | batch_start=0 | requested=200 | parsed=198 | xml_bytes=5066826


domain=cardiovascular_disease | batch_start=200 | requested=150 | parsed=150 | xml_bytes=3803799
domain=cardiovascular_disease | total_requested=350 | total_parsed=348


domain=immunotherapy_cancer | batch_start=0 | requested=200 | parsed=200 | xml_bytes=7877919


domain=immunotherapy_cancer | batch_start=200 | requested=150 | parsed=150 | xml_bytes=6621014
domain=immunotherapy_cancer | total_requested=350 | total_parsed=350
records_total=698 | unique_pmids=698 | empty_abstract=19
abstract_words_min=0 | max=376 | under_50=83
csv=data\raw\efetch\records_raw.csv | csv_bytes=711614


In [4]:
# Cell 4 - Post-retrieval filtering and domain balancing

import csv
from collections import Counter
from pathlib import Path
from statistics import median

RAW_PATH = Path("data/raw/efetch/records_raw.csv")
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

MIN_ABSTRACT_WORDS = 50
TARGET_PER_DOMAIN = 250
FIELDS = [
    "pmid",
    "domain",
    "title",
    "abstract",
    "abstract_words",
    "journal",
    "pub_year",
    "pub_types",
]

with RAW_PATH.open(encoding="utf-8") as handle:
    raw_records = list(csv.DictReader(handle))

seen_pmids = set()
valid_records = []
dropped = {"duplicate": 0, "empty_abstract": 0, "short_abstract": 0}
for record in raw_records:
    words = len(record["abstract"].split())
    if record["pmid"] in seen_pmids:
        dropped["duplicate"] += 1
        continue
    seen_pmids.add(record["pmid"])
    if words == 0:
        dropped["empty_abstract"] += 1
        continue
    if words < MIN_ABSTRACT_WORDS:
        dropped["short_abstract"] += 1
        continue
    record["abstract_words"] = words
    valid_records.append(record)

balanced = []
for domain in sorted({record["domain"] for record in valid_records}):
    pool = [record for record in valid_records if record["domain"] == domain]
    kept = pool[:TARGET_PER_DOMAIN]
    balanced.extend(kept)
    print(
        f"domain={domain} | valid_pool={len(pool)} | kept={len(kept)}"
        f" | truncated={len(pool) - len(kept)}"
    )

counts = Counter(record["domain"] for record in balanced)
assert len(balanced) == 2 * TARGET_PER_DOMAIN, f"expected 500 records, got {len(balanced)}"
assert all(count == TARGET_PER_DOMAIN for count in counts.values()), counts
assert len({record["pmid"] for record in balanced}) == len(balanced), "duplicate PMID survived"

out_path = PROCESSED_DIR / "corpus_filtered.csv"
with out_path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=FIELDS)
    writer.writeheader()
    writer.writerows(balanced)

words = [record["abstract_words"] for record in balanced]
print(f"dropped={dropped}")
print(f"kept_total={len(balanced)} | per_domain={dict(counts)} | unique_pmids={len({r['pmid'] for r in balanced})}")
print(f"abstract_words_min={min(words)} | median={median(words)} | max={max(words)}")
print(f"csv={out_path} | csv_bytes={out_path.stat().st_size}")


domain=cardiovascular_disease | valid_pool=281 | kept=250 | truncated=31
domain=immunotherapy_cancer | valid_pool=334 | kept=250 | truncated=84
dropped={'duplicate': 0, 'empty_abstract': 19, 'short_abstract': 64}
kept_total=500 | per_domain={'cardiovascular_disease': 250, 'immunotherapy_cancer': 250} | unique_pmids=500
abstract_words_min=50 | median=121.0 | max=376
csv=data\processed\corpus_filtered.csv | csv_bytes=553977


In [5]:
# Cell 5 - Stage 2 text cleaning

import csv
import html
import re
import unicodedata
from pathlib import Path
from statistics import median

IN_PATH = Path("data/processed/corpus_filtered.csv")
OUT_PATH = Path("data/processed/corpus_clean.csv")

SECTION_LABELS = re.compile(
    r"\b(?:PURPOSE|EXPERT|AREAS?|SUMMARY|KEY|UNLABELLED|OBSERVATIONS?|STATEMENT|CLINICALTRIALSGOV|"
    r"WHAT|BACKGROUND|OBJECTIVES?|METHODS?|RESULTS?|CONCLUSIONS?|INTRODUCTION|AIMS?|DESIGN|SETTING|"
    r"PARTICIPANTS?|INTERVENTIONS?|MEASUREMENTS?|MAIN|OUTCOMES?|FINDINGS|IMPORTANCE|EVIDENCE|"
    r"DISCUSSION|LIMITATIONS?|INTERPRETATION|RELEVANCE|PATIENTS?|MATERIALS|HIGHLIGHTS|IMPLICATIONS)"
    r"(?:[\s-]+(?:OF|AND|THE|FOR|IS|WHAT|[A-Z][A-Z-]+)){0,4}\s*:\s*"
)
TAG_PATTERN = re.compile(r"<[^>]+>")
URL_PATTERN = re.compile(r"\b(?:https?://|www\.)\S+")
CITATION_PATTERN = re.compile(r"\[\s*\d+(?:\s*[-,]\s*\d+)*\s*\]")
ENTITY_PATTERN = re.compile(r"&(?:[A-Za-z]+|#\d+);")
HYPHEN_TOKEN = re.compile(r"\b[A-Za-z0-9]+(?:-[A-Za-z0-9]+)+\b")
RESIDUAL_LABEL = re.compile(r"\b[A-Z][A-Z ]{3,}:")
NUMERIC_DASH = re.compile(r"(?<=\d)[\u2013\u2014](?=\d)")
DASH_TABLE = str.maketrans({"\u2010": "-", "\u2011": "-", "\u2012": "-", "\u2212": "-"})
SPACE_TABLE = str.maketrans(
    {"\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"', "\u00a0": " ", "\u200b": "", "\u00ad": ""}
)


def clean_text(raw):
    text = html.unescape(html.unescape(raw))
    text = unicodedata.normalize("NFKC", text)
    text = NUMERIC_DASH.sub("-", text)
    text = text.replace("\u2014", " ").replace("\u2013", " ")
    text = text.translate(DASH_TABLE).translate(SPACE_TABLE)
    text = TAG_PATTERN.sub(" ", text)
    text = URL_PATTERN.sub(" ", text)
    text = CITATION_PATTERN.sub(" ", text)
    text = SECTION_LABELS.sub(" ", text)
    text = "".join(char for char in text if unicodedata.category(char)[0] != "C")
    return re.sub(r"\s+", " ", text).strip()


with IN_PATH.open(encoding="utf-8") as handle:
    rows = list(csv.DictReader(handle))

hyphens_before = []
example = None
for row in rows:
    hyphens_before.extend(HYPHEN_TOKEN.findall(f"{row['title']} {row['abstract']}"))

    title_clean = clean_text(row["title"])
    abstract_clean = clean_text(row["abstract"])
    if title_clean.endswith((".", "?", "!")):
        text_clean = f"{title_clean} {abstract_clean}"
    else:
        text_clean = f"{title_clean}. {abstract_clean}"

    row["title_clean"] = title_clean
    row["text_clean"] = text_clean
    row["clean_words"] = len(text_clean.split())

    if example is None and SECTION_LABELS.search(row["abstract"]):
        example = (row["abstract"], abstract_clean)

texts = [row["text_clean"] for row in rows]
hyphens_after = []
for text in texts:
    hyphens_after.extend(HYPHEN_TOKEN.findall(text))
lost_hyphens = set(hyphens_before) - set(hyphens_after)

journal_prefixes = {"JACC:"}
residual = [
    match for text in texts for match in RESIDUAL_LABEL.findall(text) if match not in journal_prefixes
]

assert len(rows) == 500, f"expected 500 rows, got {len(rows)}"
assert all(row["clean_words"] > 0 for row in rows), "empty cleaned text"
assert sum(len(ENTITY_PATTERN.findall(text)) for text in texts) == 0, "entity survived"
assert sum(len(TAG_PATTERN.findall(text)) for text in texts) == 0, "tag survived"
assert sum(len(URL_PATTERN.findall(text)) for text in texts) == 0, "url survived"
assert not lost_hyphens, f"hyphen tokens lost: {sorted(lost_hyphens)[:5]}"
assert not residual, f"section label survived: {sorted(set(residual))}"

FIELDS = [
    "pmid",
    "domain",
    "title",
    "abstract",
    "title_clean",
    "text_clean",
    "abstract_words",
    "clean_words",
    "journal",
    "pub_year",
    "pub_types",
]
with OUT_PATH.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=FIELDS)
    writer.writeheader()
    writer.writerows(rows)

word_counts = [row["clean_words"] for row in rows]
print(
    f"records={len(rows)} | empty_text=0 | hyphen_tokens_before={len(hyphens_before)}"
    f" | after={len(hyphens_after)} | created_by_dash_normalization={len(hyphens_after) - len(hyphens_before)}"
)
print(f"clean_words_min={min(word_counts)} | median={median(word_counts)} | max={max(word_counts)}")
print(f"residual_section_labels={len(residual)} | journal_prefix_kept=JACC:")
if example is not None:
    print(f"label_strip_before={example[0][:110]}")
    print(f"label_strip_after={example[1][:110]}")
print(f"csv={OUT_PATH} | csv_bytes={OUT_PATH.stat().st_size}")


records=500 | empty_text=0 | hyphen_tokens_before=1191 | after=1194 | created_by_dash_normalization=3
clean_words_min=60 | median=131.0 | max=379
residual_section_labels=0 | journal_prefix_kept=JACC:
label_strip_before=PURPOSE OF REVIEW: In this review, we synthesise recent research on the association between loneliness and car
label_strip_after=In this review, we synthesise recent research on the association between loneliness and cardiovascular disease
csv=data\processed\corpus_clean.csv | csv_bytes=1097259


In [6]:
# Cell 6 - Stage 3 biomedical preprocessing with scispaCy

import csv
import re
import warnings
from pathlib import Path
from statistics import median

import spacy
from spacy.lang.en.stop_words import STOP_WORDS

CORPUS_PATH = Path("data/processed/corpus_clean.csv")

# en_core_sci_sm 0.5.4 ships config values as strings ("False"), which spaCy 3.8
# rejects; the installed config.cfg was corrected to false before this cell ran.
DOMAIN_STOPWORDS = {
    "study", "studies", "patient", "patients", "result", "results", "method", "methods",
    "conclusion", "conclusions", "background", "objective", "objectives", "aim", "aims",
    "group", "groups", "data", "analysis", "analyses", "significant", "significantly",
    "compared", "comparison", "outcome", "outcomes", "review", "reviews", "trial", "trials",
    "cohort", "article", "paper", "report", "reported", "reports", "included", "including",
}
STOPWORD_SET = set(STOP_WORDS) | DOMAIN_STOPWORDS
HYPHEN_TOKEN = re.compile(r"\b[A-Za-z0-9]+(?:-[A-Za-z0-9]+)+\b")

warnings.filterwarnings("ignore", message=".*trained with spaCy.*")
nlp = spacy.load("en_core_sci_sm", disable=["parser", "ner"])


def to_lemma_text(doc):
    lemmas = []
    for token in doc:
        lemma = token.lemma_.strip().lower()
        if not lemma or token.is_punct or token.is_space:
            continue
        if lemma in STOPWORD_SET:
            continue
        if not any(char.isalpha() for char in lemma):
            continue
        lemmas.append(lemma)
    return " ".join(lemmas)


with CORPUS_PATH.open(encoding="utf-8") as handle:
    rows = list(csv.DictReader(handle))

docs = nlp.pipe((row["text_clean"] for row in rows), batch_size=32)
for row, doc in zip(rows, docs):
    row["text_lemma"] = to_lemma_text(doc)
    row["lemma_words"] = len(row["text_lemma"].split())

lemma_counts = [row["lemma_words"] for row in rows]
vocabulary = {token for row in rows for token in row["text_lemma"].split()}
hyphens_before = {token.lower() for row in rows for token in HYPHEN_TOKEN.findall(row["text_clean"])}
hyphens_after = {token for row in rows for token in HYPHEN_TOKEN.findall(row["text_lemma"])}
lost_hyphens = sorted(hyphens_before - hyphens_after)

assert len(rows) == 500, f"expected 500 rows, got {len(rows)}"
assert all(count > 0 for count in lemma_counts), "empty lemma text"
assert all(row["pmid"] for row in rows), "missing pmid"

FIELDS = [
    "pmid",
    "domain",
    "title",
    "abstract",
    "title_clean",
    "text_clean",
    "text_lemma",
    "abstract_words",
    "clean_words",
    "lemma_words",
    "journal",
    "pub_year",
    "pub_types",
]
with CORPUS_PATH.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=FIELDS)
    writer.writeheader()
    writer.writerows(rows)

example = rows[0]
print(f"records={len(rows)} | pipeline={nlp.pipe_names}")
print(
    f"lemma_words_min={min(lemma_counts)} | median={median(lemma_counts)}"
    f" | max={max(lemma_counts)} | vocabulary={len(vocabulary)}"
)
print(
    f"hyphen_tokens_before={len(hyphens_before)} | after={len(hyphens_after)}"
    f" | lost={len(lost_hyphens)} | lost_sample={lost_hyphens[:5]}"
)
print(f"example_pmid={example['pmid']} | clean={example['text_clean'][:90]}")
print(f"example_lemma={example['text_lemma'][:90]}")
print(f"csv={CORPUS_PATH} | csv_bytes={CORPUS_PATH.stat().st_size}")


C:\Users\Hi\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\spacy\language.py:2235: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


records=500 | pipeline=['tok2vec', 'tagger', 'attribute_ruler', 'lemmatizer']
lemma_words_min=31 | median=78.5 | max=248 | vocabulary=4895
hyphen_tokens_before=613 | after=599 | lost=34 | lost_sample=['10-14', '10-30', '1990-2021', '2-3', '2001-2019']
example_pmid=37455897 | clean=Sarcopenic obesity: epidemiology, pathophysiology, cardiovascular disease, mortality, and 
example_lemma=sarcopenic obesity epidemiology pathophysiology cardiovascular disease mortality managemen
csv=data\processed\corpus_clean.csv | csv_bytes=1459821


In [7]:
# Cell 7 - Export handover corpus

from pathlib import Path

import pandas as pd

SOURCE_PATH = Path("data/processed/corpus_clean.csv")
CSV_PATH = Path("data/processed/corpus_handover.csv")

TEXT_COLUMNS = ["pmid", "domain", "title", "abstract", "title_clean", "text_clean", "text_lemma"]
COUNT_COLUMNS = ["abstract_words", "clean_words", "lemma_words"]
META_COLUMNS = ["journal", "pub_year", "pub_types"]

df = pd.read_csv(SOURCE_PATH, dtype=str, keep_default_na=False)
for column in COUNT_COLUMNS:
    df[column] = df[column].astype("int32")
df["pub_year"] = pd.to_numeric(df["pub_year"], errors="coerce").astype("Int32")
df = df[TEXT_COLUMNS + COUNT_COLUMNS + META_COLUMNS]

assert len(df) == 500, f"expected 500 rows, got {len(df)}"
assert df["pmid"].is_unique, "duplicate pmid"
counts = df["domain"].value_counts().to_dict()
assert counts == {"cardiovascular_disease": 250, "immunotherapy_cancer": 250}, counts
assert (df["text_clean"].str.len() > 0).all(), "empty text_clean"
assert (df["text_lemma"].str.len() > 0).all(), "empty text_lemma"
assert not df[TEXT_COLUMNS].isna().any().any(), "null in text columns"

df.to_csv(CSV_PATH, index=False, encoding="utf-8")

readback = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)
assert len(readback) == len(df), "csv row count mismatch"
assert list(readback.columns) == list(df.columns), "csv column order mismatch"
assert readback["pmid"].nunique() == 500, "csv pmid mismatch"

print(f"rows={len(df)} | columns={len(df.columns)} | per_domain={counts}")
print(f"dtypes={ {name: str(dtype) for name, dtype in df.dtypes.items()} }")
print(f"pub_year_missing={int(df['pub_year'].isna().sum())} | pub_year_range={int(df['pub_year'].min())} to {int(df['pub_year'].max())}")
print(f"csv={CSV_PATH} | bytes={CSV_PATH.stat().st_size}")
print(f"csv_readback_rows={len(readback)} | unique_pmids={readback['pmid'].nunique()}")


rows=500 | columns=13 | per_domain={'cardiovascular_disease': 250, 'immunotherapy_cancer': 250}
dtypes={'pmid': 'str', 'domain': 'str', 'title': 'str', 'abstract': 'str', 'title_clean': 'str', 'text_clean': 'str', 'text_lemma': 'str', 'abstract_words': 'int32', 'clean_words': 'int32', 'lemma_words': 'int32', 'journal': 'str', 'pub_year': 'Int32', 'pub_types': 'str'}
pub_year_missing=0 | pub_year_range=2020 to 2026
csv=data\processed\corpus_handover.csv | bytes=1459821
csv_readback_rows=500 | unique_pmids=500
